In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as func

device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 64
batch_size = 128

max_iterations = 3000
learning_rate = 3e-4
eval_interval = 100

n_embed = 192
n_head = 8
n_layer = 8

dropout = 0.2

In [13]:
vocab = ""
with open("moby-dick.txt", "r", encoding='utf-8') as f:
    text = f.read()
    vocab = sorted(list(set(text)))

vocab_size = len(vocab)

In [14]:
string_to_int = { ch:i for i,ch in enumerate(vocab) }
int_to_string = { i:ch for i,ch in enumerate(vocab) }

encode = lambda s: [string_to_int[c] for c in s ]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [15]:
n = int(0.8 * len(data))

train_data = data[:n]
validate_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validate_data
    
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [16]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    for split in ['train', 'validate']:
        losses = torch.zeros(eval_interval)

        for k in range(eval_interval):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()
    
    model.train()
    return out

In [17]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        BATCH, TIME, CHANNEL = x.shape
        
        key = self.key(x)
        query = self.query(x)

        weights = query @ key.transpose(-2, -1) * key.shape[-1] ** -0.5
        weights = weights.masked_fill(self.tril[:TIME, :TIME] == 0, float('-inf'))
        weights = func.softmax(weights, dim=-1)
        weights = self.dropout(weights)

        value = self.value(x)
        out = weights @ value

        return out

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.project = nn.Linear(head_size * n_head, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.project(out))
        
        return out

In [19]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [20]:
class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()

        head_size = n_embed // n_head

        self.self_attention = MultiHeadAttention(n_head, head_size)
        self.feed_fwd = FeedForward(n_embed)
        self.layer_norm1 = nn.LayerNorm(n_embed)
        self.layer_norm2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        y = self.self_attention(x)
        x = self.layer_norm1(x + y)
        y = self.feed_fwd(x)
        x = self.layer_norm2(x + y)

        return x

In [21]:
class GPTLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.token_embed_table = nn.Embedding(vocab_size, n_embed)
        self.pos_embed_table   = nn.Embedding(block_size, n_embed)
        
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head) for _ in range(n_layer)])

        self.layer_norm_final = nn.LayerNorm(n_embed)
        self.lang_model_head = nn.Linear(n_embed, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.2)
    
    def forward(self, index, targets=None):
        BATCH, TIME = index.shape
        
        token_embed = self.token_embed_table(index)
        pos_embed = self.pos_embed_table(torch.arange(TIME, device=device))

        x = token_embed + pos_embed
        x = self.blocks(x)
        x = self.layer_norm_final(x)

        logits = self.lang_model_head(x)
        
        if targets == None:
            loss = None
        else:
            BATCH, TIME, CHANNEL = logits.shape
            logits = logits.view(BATCH * TIME, CHANNEL)
            targets = targets.view(BATCH * TIME)
            loss = func.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            
            logits = logits[:, -1, :]
            probs = func.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)

        return index

model = GPTLM(vocab_size)
m = model.to(device)

In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iterations):
    if iteration % eval_interval == 0:
        losses = estimate_loss()
        print(f'iteration: {iteration}, loss {losses}')
    
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

iteration: 0, loss {'train': tensor(4.5470), 'validate': tensor(4.5477)}
iteration: 100, loss {'train': tensor(2.5413), 'validate': tensor(2.5534)}
iteration: 200, loss {'train': tensor(2.3652), 'validate': tensor(2.3732)}
iteration: 300, loss {'train': tensor(2.2177), 'validate': tensor(2.2245)}
iteration: 400, loss {'train': tensor(2.0779), 'validate': tensor(2.0879)}
iteration: 500, loss {'train': tensor(1.9764), 'validate': tensor(1.9919)}
iteration: 600, loss {'train': tensor(1.9091), 'validate': tensor(1.9271)}
iteration: 700, loss {'train': tensor(1.8485), 'validate': tensor(1.8688)}
iteration: 800, loss {'train': tensor(1.8091), 'validate': tensor(1.8286)}
iteration: 900, loss {'train': tensor(1.7635), 'validate': tensor(1.7952)}
iteration: 1000, loss {'train': tensor(1.7341), 'validate': tensor(1.7729)}
iteration: 1100, loss {'train': tensor(1.7060), 'validate': tensor(1.7393)}
iteration: 1200, loss {'train': tensor(1.6797), 'validate': tensor(1.7220)}
iteration: 1300, loss {'